# GLORIA parser walkthrough

This notebook is the practical guide for parsing GLORIA monetary SUT releases in MARIO.

## What this notebook covers

- where the GLORIA release material comes from;
- how the local GLORIA release should be laid out on disk;
- when to pass the release root and when to pass the `GLORIA_MRIOs_*` directory directly;
- how `valuation=`, `regions=`, and `satellites=` change the parse;
- why `dtype` and `cache` matter for large GLORIA runs.

## Relevant source pages

- GLORIA overview: [IELab GLORIA overview](https://ielab.info/resources/gloria/about)
- release notes and supporting material: [IELab GLORIA supporting documents](https://ielab.info/resources/gloria/supportingdocs)

MARIO does not download GLORIA automatically. The expected workflow is to obtain the release locally and then point the parser to the release root or directly to the `GLORIA_MRIOs_*` directory.

## Main entry point

For normal user workflows, the public entry point is:

- `mario.parse_gloria(...)`

The current backend supports monetary `SUT` parsing only.

## Key arguments

The key public arguments are:

- `path`: GLORIA release root or directly the `GLORIA_MRIOs_*` directory;
- `table`: currently only `"SUT"` is supported;
- `valuation`: choose one markup branch such as `basic`, `trade`, `transport`, `taxes`, or `subsidies`;
- `year`: used when the selected root contains more than one GLORIA year;
- `regions`: optional subset of GLORIA region acronyms;
- `satellites`: optional satellite group or row selector;
- `dtype`: numeric storage type, with `float32` as the practical default;
- `cache`: `True` or one explicit path to persist the parsed result.

## Local layout expectation

The parser expects the local GLORIA release to include:

- the raw `T`, `Y`, and `V` csv files inside one `GLORIA_MRIOs_*` directory;
- the matching `GLORIA_ReadMe_*.xlsx` workbook next to that dataset;
- optionally, the `GLORIA_SatelliteAccounts_*` directory when you want actual satellite accounts instead of empty placeholders.

You can point MARIO either to the release root or directly to the `GLORIA_MRIOs_*` directory.

In [ ]:
import mario

## Parse one GLORIA SUT

Use this when you want the default `basic` valuation and the full region set.

In [ ]:
db = mario.parse_gloria(
    path="/path/to/gloria_release",
    table="SUT",
    calc_all=False,
)

db

## Select one valuation branch and one region subset

This is often the right default when you do not need the full release in memory.

In [ ]:
db = mario.parse_gloria(
    path="/path/to/gloria_release",
    table="SUT",
    valuation="trade",
    regions=["ITA", "DEU", "FRA"],
    calc_all=False,
)

db

## Restrict the satellite payload

You can keep all satellites, one full row label, or one group name from the GLORIA ReadMe.

In [ ]:
db = mario.parse_gloria(
    path="/path/to/gloria_release",
    table="SUT",
    satellites="Emissions",
    calc_all=False,
)

db

## Cache repeated parses

GLORIA files are large. If you revisit the same release repeatedly, caching usually saves a lot of time.

In [ ]:
db = mario.parse_gloria(
    path="/path/to/gloria_release",
    table="SUT",
    cache=True,
    dtype="float32",
    calc_all=False,
)

db

## Practical warnings

- `GLORIA` is currently `SUT`-only in MARIO.
- Full-region parses can be memory-intensive; `regions=`, `dtype="float32"`, and `cache=True` are usually sensible defaults.
- If the local release does not include a complete satellite-account directory, MARIO falls back to empty extensions.